# Fase 3 — Preparação dos Dados

O objetivo desta fase é transformar os dados brutos em um formato adequado para os algoritmos. O guia organiza o trabalho em quatro etapas:

**Selecionar (filtrar) → Limpar (tratar) → Construir (derivar) → Integrar (mesclar)**

Cada etapa abaixo mostra a regra aplicada e a conferência do resultado. As mesmas regras estão reunidas na função `preparar_dados` do módulo `src/preparacao.py`, para que a preparação seja idêntica aqui e na pontuação de novos clientes (Fase 6).

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

RAIZ = Path("..").resolve()
sys.path.append(str(RAIZ))
pd.set_option("display.max_columns", 30)

bruto = pd.read_csv(RAIZ / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = bruto.copy()
print(df.shape)

(7043, 21)


## 1. Selecionar

**Linhas:** todas as 7.043 são mantidas. Os 11 clientes com `tenure` = 0 são clientes reais recém-chegados, e não erros de registro.

**Colunas:**

| Decisão | Colunas | Motivo |
|---|---|---|
| Manter como atributos | As 19 colunas de perfil, serviços, contrato e cobrança | Todas estão disponíveis no momento da previsão |
| Manter só para rastreio | `customerID` | Necessário para entregar a lista de clientes à equipe de retenção, mas não entra no modelo |
| Alvo | `Churn` | Variável a prever |

Nenhum atributo confidencial (como dados de cartão) existe nesta base. `gender` e `SeniorCitizen` são atributos sensíveis: ficam na base e seu peso no modelo é verificado na Fase 5.

## 2. Limpar

### 2.1 `TotalCharges`: texto para número (achado Q1)

In [2]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Valores sem número após a conversão:", int(df["TotalCharges"].isna().sum()))

df["TotalCharges"] = df["TotalCharges"].fillna(df["tenure"] * df["MonthlyCharges"])
print("Valores sem número após o preenchimento:", int(df["TotalCharges"].isna().sum()))
df.loc[df["tenure"] == 0, ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]].head()

Valores sem número após a conversão: 11
Valores sem número após o preenchimento: 0


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,0.0
753,3115-CZMZD,0,20.25,0.0
936,5709-LVOEQ,0,80.85,0.0
1082,4367-NUYAO,0,25.75,0.0
1340,1371-DWPAZ,0,56.05,0.0


Os 11 valores em branco foram preenchidos com permanência × mensalidade, que resulta em 0 para quem ainda não recebeu fatura. Preencher com a média ou a mediana distorceria o dado, porque esses clientes de fato não pagaram nada ainda.

### 2.2 Categorias redundantes (achado Q3) e padronização (achado Q2)

In [3]:
servicos = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]

df[servicos] = df[servicos].replace("No internet service", "No")
df["MultipleLines"] = df["MultipleLines"].replace("No phone service", "No")
df["SeniorCitizen"] = df["SeniorCitizen"].replace({0: "No", 1: "Yes"})

for coluna in servicos + ["MultipleLines", "SeniorCitizen"]:
    print(f"{coluna:17s} {sorted(df[coluna].unique())}")

OnlineSecurity    ['No', 'Yes']
OnlineBackup      ['No', 'Yes']
DeviceProtection  ['No', 'Yes']
TechSupport       ['No', 'Yes']
StreamingTV       ['No', 'Yes']
StreamingMovies   ['No', 'Yes']
MultipleLines     ['No', 'Yes']
SeniorCitizen     ['No', 'Yes']


A informação "não tem internet" continua disponível na coluna `InternetService`, portanto nada se perde.

## 3. Construir

Três atributos derivados foram propostos, cada um ligado a uma hipótese:

| Atributo | Como é calculado | Hipótese |
|---|---|---|
| `NumServicosAdicionais` | Quantidade de serviços adicionais contratados (0 a 6) | H5: clientes com mais serviços ficam mais |
| `FaixaTenure` | Permanência em faixas: 0-12, 13-24, 25-48 e 49+ meses | H2: o risco cai em degraus com o tempo de casa |
| `VariacaoCobranca` | Mensalidade atual − cobrança média histórica (`TotalCharges` ÷ `tenure`) | Nova: um aumento recente de preço pode motivar a saída |

In [4]:
df["NumServicosAdicionais"] = (df[servicos] == "Yes").sum(axis=1)

df["FaixaTenure"] = pd.cut(
    df["tenure"], bins=[-1, 12, 24, 48, np.inf], labels=["00-12m", "13-24m", "25-48m", "49m+"]
).astype(str)

df["Churn"] = (df["Churn"] == "Yes").astype(int)

print("Taxa de cancelamento por faixa de permanência (%):")
print((df.groupby("FaixaTenure")["Churn"].mean() * 100).round(1).to_string())
print("\nTaxa de cancelamento por número de serviços adicionais (%):")
print((df.groupby("NumServicosAdicionais")["Churn"].mean() * 100).round(1).to_string())

Taxa de cancelamento por faixa de permanência (%):
FaixaTenure
00-12m    47.4
13-24m    28.7
25-48m    20.4
49m+       9.5

Taxa de cancelamento por número de serviços adicionais (%):
NumServicosAdicionais
0    21.4
1    45.8
2    35.8
3    27.4
4    22.3
5    12.4
6     5.3


Os dois primeiros atributos separam bem os grupos. A relação com o número de serviços não é linear: clientes sem nenhum serviço adicional cancelam pouco (21,4%), porque boa parte deles não tem internet; o pico está em quem tem apenas um serviço (45,8%), e a taxa cai até 5,3% para quem tem os seis.

A variável-alvo foi convertida para 1 (cancelou) e 0 (permaneceu).

### Atributo testado e descartado: `VariacaoCobranca`

In [5]:
media_historica = np.where(df["tenure"] > 0, df["TotalCharges"] / df["tenure"].clip(lower=1), df["MonthlyCharges"])
variacao = pd.Series(df["MonthlyCharges"] - media_historica, name="VariacaoCobranca")

print(variacao.groupby(df["Churn"]).describe().round(2).to_string())
print("\nCorrelação com o cancelamento:", round(variacao.corr(df["Churn"]), 3))

        count  mean   std    min   25%  50%   75%    max
Churn                                                   
0      5174.0 -0.00  2.30 -17.12 -1.12  0.0  1.11  17.02
1      1869.0  0.01  3.33 -18.90 -1.34  0.0  1.45  19.13

Correlação com o cancelamento: 0.002


A hipótese do aumento de preço **não se confirmou**: a variação de cobrança tem média e mediana praticamente iguais a zero nos dois grupos e correlação quase nula com o cancelamento. O atributo foi descartado para não acrescentar ruído ao modelo. A base final fica com dois atributos derivados.

## 4. Integrar

O projeto tem uma única fonte, com uma linha por cliente, então **não há tabelas para mesclar**. Esta etapa do guia fica registrada como não aplicável. Em uma operadora real, seria aqui que a base de clientes se juntaria, pelo `customerID`, a tabelas de atendimento, faturas e uso dos serviços.

## 5. Conferência com a função reutilizável

A função `preparar_dados` reúne todas as regras acima. A célula abaixo confirma que ela produz exatamente a mesma tabela construída passo a passo neste notebook.

In [6]:
from src.preparacao import preparar_dados

preparado = preparar_dados(bruto)
pd.testing.assert_frame_equal(preparado, df, check_dtype=False)
print("A função preparar_dados reproduz a preparação deste notebook.")

A função preparar_dados reproduz a preparação deste notebook.


In [7]:
print("Formato final:", preparado.shape)
print("Valores nulos:", int(preparado.isna().sum().sum()))
print("Taxa de cancelamento:", round(preparado["Churn"].mean(), 4))

destino = RAIZ / "data" / "processed" / "telco_churn_preparado.csv"
preparado.to_csv(destino, index=False)
print("Arquivo salvo em:", destino.relative_to(RAIZ))
preparado.head()

Formato final: (7043, 23)
Valores nulos: 0
Taxa de cancelamento: 0.2654
Arquivo salvo em: data/processed/telco_churn_preparado.csv


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,NumServicosAdicionais,FaixaTenure
0,7590-VHVEG,Female,No,Yes,No,1,No,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0,1,00-12m
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0,2,25-48m
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,2,00-12m
3,7795-CFOCW,Male,No,No,No,45,No,No,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0,3,25-48m
4,9237-HQITU,Female,No,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,0,00-12m


## O que fica para a Fase 4

A conversão das categorias em números (one-hot encoding) e a padronização da escala dos atributos numéricos **não** são feitas aqui. Elas dependem de estatísticas calculadas sobre os dados (média, desvio, lista de categorias) e, se fossem calculadas com a base inteira, levariam informação do conjunto de teste para o treino. Por isso entram no pipeline do modelo, ajustadas somente com os dados de treino.

## Resumo da fase

| Etapa | O que foi feito |
|---|---|
| Selecionar | 7.043 linhas mantidas; `customerID` fora do modelo |
| Limpar | `TotalCharges` numérica com 11 valores preenchidos; categorias redundantes unificadas; `SeniorCitizen` padronizada |
| Construir | 2 atributos derivados mantidos e 1 testado e descartado; alvo em 0/1 |
| Integrar | Não aplicável (fonte única) |
| Saída | `data/processed/telco_churn_preparado.csv`, com 7.043 linhas e 23 colunas |